# LLM 입력을 위한 최종 프롬프트 생성 노트북 (v3 - 수정판)

**수정 사항:** 제품 카테고리와 시장 데이터 카테고리 간의 매칭 로직을 개선하여 모든 프롬프트에 시장 정보가 정상적으로 포함되도록 수정했습니다.

In [ ]:
import json
import pandas as pd

# --- 파일 경로 설정 ---
PRODUCT_FILE = 'product_info_final.jsonl'
PERSONA_FILE = 'persona_attributes_weighted.jsonl'
COMPETITOR_FILE_CLEANED = 'competitor_prices_cleaned_final.csv'
OUTPUT_FILE = 'prompts_for_llm.jsonl'

## 1. 경쟁사 시장 데이터 불러오기

In [ ]:
def load_market_context(filepath):
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"오류: '{filepath}' 파일을 찾을 수 없습니다.")
        return None

    market_context = {}
    if 'category' in df.columns:
        for category, group in df.groupby('category'):
            min_price = group['price_per_100g'].min()
            max_price = group['price_per_100g'].max()
            market_context[category] = {
                "competitor_price_range_per_100g": f"{int(min_price):,}원 ~ {int(max_price):,}원"
            }
    return market_context

print("1. 전처리된 경쟁사 가격 데이터를 불러옵니다...")
market_context = load_market_context(COMPETITOR_FILE_CLEANED)
if market_context is not None:
    print("   => 완료. 카테고리별 시장 데이터가 준비되었습니다.")
    print(f"   (감지된 시장 카테고리: {list(market_context.keys())})")

## 2. 제품 및 페르소나 데이터 불러오기

In [ ]:
print(f"2. '{PRODUCT_FILE}'와 '{PERSONA_FILE}' 파일을 불러옵니다...")
with open(PRODUCT_FILE, 'r', encoding='utf-8') as f:
    products = [json.loads(line) for line in f]
with open(PERSONA_FILE, 'r', encoding='utf-8') as f:
    personas = [json.loads(line) for line in f]
print(f"   => 완료. 제품 {len(products)}개, 페르소나 {len(personas)}개를 불러왔습니다.")

## 3. 프롬프트 생성 함수 정의 (로직 수정)

In [ ]:
def map_to_main_category(detailed_category):
    """
    [수정] 키워드 기반으로 제품의 상세 카테고리를 대표 카테고리에 매핑합니다.
    """
    # 키워드 : 대표 카테고리
    mapping_rules = {
        '참치캔': '참치캔',
        '참치액': '참치액',
        '액상조미료': '참치액',
        '그릭': '그릭요거트',
        '발효유': '그릭요거트',
        '캔햄': '캔햄',
        '햄': '캔햄',
        '커피': 'RTD_액상커피'
    }
    for keyword, main_category in mapping_rules.items():
        if keyword in detailed_category:
            return main_category
    return None
def create_single_prompt(product_info, persona_info, market_context):
    """
    제품, 페르소나, 시장 정보를 바탕으로 최종 프롬프트를 생성합니다.
    """
    product_str = json.dumps(product_info, ensure_ascii=False, indent=4)
    persona_str = json.dumps(persona_info, ensure_ascii=False, indent=4)

    # [수정] 새로운 매핑 함수를 사용하여 시장 정보를 가져옵니다.
    detailed_category = product_info.get('category', '')
    main_category_key = map_to_main_category(detailed_category)
    
    context_data = market_context.get(main_category_key, {})
    context_str = json.dumps(context_data, ensure_ascii=False, indent=4) if context_data else "{}"

    # 4. 최종 프롬프트 템플릿 (이하 동일)
    prompt_template = f"""# ROLE
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 \"제품 정보\", 원시 \"페르소나 데이터\", 그리고 \"시장 경쟁 환경\"을 종합적으로 분석하여, 페르소나가 해당 제품의 잠재 구매자로서 어떤 특징을 보일지 예측하고, 그 결과를 하나의 완결된 JSON 객체로 생성하는 것입니다.

# INSTRUCTION
아래의 모든 정보를 바탕으로, 페르소나가 해당 제품의 구매자로서 성립하는 **싱글턴 페르소나 JSON**을 생성하세요. 페르소나의 속성(attributes)과 제품의 특징(features)을 논리적으로 연결하여 구매 확률과 이유, 월별 구매 빈도를 예측해야 합니다.

# INPUT DATA
## 1. 제품 정보
{product_str}

## 2. 페르소나 데이터
{persona_str}

## 3. 시장 경쟁 환경
{context_str}

# OUTPUT FORMAT
반드시 아래와 같은 구조의 JSON 형식으로만 응답하세요. 다른 설명은 추가하지 마세요.

{{{{
  \"product_name\": \"{product_info.get('product_name', '')}\",
  \"persona_key\": {persona_info.get('persona_key')},
  \"purchase_behavior_prediction\": {{{{
    \"purchase_probability_pct\": \"<여기에 구매 확률(0-100)을 숫자로 예측>\",
    \"reason\": \"<여기에 페르소나 속성과 제품 특징, 시장 상황을 연결한 구매 결정 이유를 상세히 서술>\",
    \"monthly_purchase_frequency\": {{{{
      \"2024-07\": 0, \"2024-08\": 0, \"2024-09\": 0, \"2024-10\": 0, \"2024-11\": 0, \"2024-12\": 0,
      \"2025-01\": 0, \"2025-02\": 0, \"2025-03\": 0, \"2024-04\": 0, \"2025-05\": 0, \"2025-06\": 0
    }}}}
  }}}}
}}}} """
    return prompt_template.strip()

print("수정된 프롬프트 생성 함수가 정의되었습니다.")

## 4. 모든 조합에 대한 프롬프트 생성 및 저장

In [ ]:
if 'market_context' in locals() and market_context is not None:
    print("3. 모든 조합에 대한 프롬프트 생성을 시작합니다...")
    all_prompts = []
    for product in products:
        for persona in personas:
            product_essentials = {k: product.get(k) for k in ['brand', 'product_name', 'category', 'features', 'targeted_consumer', 'price_text', 'advertise_info']}
            prompt = create_single_prompt(product_essentials, persona, market_context)
            all_prompts.append({
                "product_name": product.get("product_name"),
                "persona_key": persona.get("persona_key"),
                "prompt": prompt
            })

    print(f"\n4. 생성된 프롬프트를 '{OUTPUT_FILE}' 파일로 저장합니다...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in all_prompts:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("="*40)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print(f"총 {len(all_prompts)}개의 프롬프트가 생성되어 '{OUTPUT_FILE}'에 저장되었습니다.")
else:
    print("시장 데이터(market_context)가 로드되지 않아 프롬프트 생성을 건너뜁니다.")